In [12]:
# Importar librerías
import pandas as pd
import numpy as np
import json
import os
import csv
import torch

## Cargar dataset

In [27]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
from datasets import load_dataset

path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
dataset_anx = load_dataset("csv", data_files={
    "train": path + "anxiety_train.csv",
    "test": path + "anxiety_test.csv",
    "validation": path + "anxiety_val.csv"
})

## Cargar DistilBETO

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dccuchile/distilbert-base-spanish-uncased"  # DistilBETO

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/530 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/269M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dccuchile/distilbert-base-spanish-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenizar el dataset

In [19]:
def tokenize(batch):
    return tokenizer(
        batch["texto"],
        padding="max_length",
        truncation=True,
        max_length=512    # Para DistilBETO; BigBird será 1024–4096
    )

encoded_dataset = dataset_anx.map(tokenize, batched=True)
encoded_dataset = encoded_dataset.rename_column("bs", "labels")
encoded_dataset = encoded_dataset.remove_columns(["texto", "user_id"])
encoded_dataset.set_format("torch")

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

## Calcular los pesos de las clases

In [21]:
from collections import Counter

# Extraer labels como lista de Python
labels = [int(x) for x in encoded_dataset["train"]["labels"]]

# Contar frecuencia por clase
class_counts = Counter(labels)
print("Frecuencia de clases:", class_counts)

num_classes = len(class_counts)

# Pesos inversamente proporcionales a la frecuencia
class_weights = np.array(
    [1.0 / class_counts[i] for i in range(num_classes)],
    dtype=np.float32
)

# Normalizar (opcional pero recomendable)
class_weights = class_weights / class_weights.sum() * num_classes

class_weights = torch.tensor(class_weights)
print("Pesos de clase:", class_weights)

Frecuencia de clases: Counter({1: 310, 0: 40})
Pesos de clase: tensor([1.7714, 0.2286])


## Fine-tuning del modelo

In [23]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

# Crear un Trainer personalizado con loss ponderada
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Definir loss ponderada
        loss_func = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_func(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Configurar el entrenamiento
training_args = TrainingArguments(
    output_dir='distilbeto-finetuned',   # Carpeta donde se guardarán los modelos y resultados
    eval_strategy="epoch",               # Evaluar al final de cada época
    save_strategy="epoch",               # Guardar un checkpoint al final de cada época
    load_best_model_at_end=True,         # Cargar automáticamente el mejor modelo al finalizar el entrenamiento
    metric_for_best_model="f1",          # Usar la métrica F1 para decidir cuál es el mejor modelo
    greater_is_better=True,              # Un valor más alto de F1 indica un mejor modelo

    num_train_epochs=3,                  # Número total de épocas de entrenamiento
    per_device_train_batch_size=16,      # Tamaño de batch para entrenamiento
    per_device_eval_batch_size=64,       # Tamaño de batch para evaluación (puede ser mayor)

    warmup_ratio = 0.1,                   # Porcentaje del entrenamiento dedicado al calentamiento para estabilizar la tasa de aprendizaje
    weight_decay=0.01,                   # Regularización L2 para evitar sobreajuste

    save_total_limit=1,                  # Solo conservar 1 checkpoint (ahorra espacio)
    fp16=True,                           # Entrenamiento en precisión mixta (más rápido y menor uso de VRAM)

    report_to="none"
)

# Definir métricas de evaluación
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }

# Entrenar el modelo
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

trainer.train()

/tmp/ipython-input-3064715743.py:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.621798,0.560000,0.499899
2,No log,0.546664,0.920000,0.728261
3,No log,0.383106,0.933333,0.834144


TrainOutput(global_step=66, training_loss=0.5304494337602095, metrics={'train_runtime': 54.7414, 'train_samples_per_second': 19.181, 'train_steps_per_second': 1.206, 'total_flos': 139090768588800.0, 'train_loss': 0.5304494337602095, 'epoch': 3.0})

## Evaluar en el conjunto de test

In [29]:
# Obtener los logits, labels reales y métricas
predictions_output = trainer.predict(encoded_dataset["test"])

# Logits -> predicciones
logits = predictions_output.predictions
preds = np.argmax(logits, axis=1)

# Añadir predicciones al dataset original de test
test_df = dataset_anx["test"].to_pandas()
test_df["pred"] = preds

output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/resultados/distilbeto_anx.csv"
test_df.to_csv(output_path, index=False)

### Comprobación de predicciones

In [31]:
# Etiquetas verdaderas
true_labels = [int(x) for x in encoded_dataset["test"]["labels"]]

f1_test = f1_score(true_labels, preds, average="macro")

# F1 ponderada (tiene en cuenta el desbalance)
f1_weighted = f1_score(true_labels, preds, average="weighted")

# F1 micro (equivalente a accuracy en multiclase)
f1_micro = f1_score(true_labels, preds, average="micro")

print("F1 macro:", f1_test)
print("F1 weighted:", f1_weighted)
print("F1 micro:", f1_micro)

F1 macro: 0.6685415829650462
F1 weighted: 0.8566331860184812
F1 micro: 0.8533333333333334


## Guardar el modelo final

In [30]:
from google.colab import files
!zip -r distilbeto-finetuned.zip distilbeto-finetuned
files.download("distilbeto-finetuned.zip")

  adding: distilbeto-finetuned/ (stored 0%)
  adding: distilbeto-finetuned/checkpoint-66/ (stored 0%)
  adding: distilbeto-finetuned/checkpoint-66/training_args.bin (deflated 53%)
  adding: distilbeto-finetuned/checkpoint-66/trainer_state.json (deflated 64%)
  adding: distilbeto-finetuned/checkpoint-66/tokenizer.json (deflated 71%)
  adding: distilbeto-finetuned/checkpoint-66/vocab.txt (deflated 56%)
  adding: distilbeto-finetuned/checkpoint-66/special_tokens_map.json (deflated 80%)
  adding: distilbeto-finetuned/checkpoint-66/model.safetensors (deflated 7%)
  adding: distilbeto-finetuned/checkpoint-66/tokenizer_config.json (deflated 75%)
  adding: distilbeto-finetuned/checkpoint-66/scheduler.pt (deflated 62%)
  adding: distilbeto-finetuned/checkpoint-66/config.json (deflated 46%)
  adding: distilbeto-finetuned/checkpoint-66/rng_state.pth (deflated 26%)
  adding: distilbeto-finetuned/checkpoint-66/optimizer.pt (deflated 30%)
  adding: distilbeto-finetuned/checkpoint-66/scaler.pt (defla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>